# Phase 1 — Step 4: Chunking (with metadata on every chunk)

Split the parsed FSR document into retrieval-sized pieces, and **attach metadata to each chunk** so downstream stages (filter, retrieve, cite) have what they need.

**Why chunking matters (cert + interview):**
- Embedding models have a token cap (and quality drops well before it).
- Retrieval returns *chunks*, not docs — chunk size = your retrieval granularity.
- Too small → context gets fragmented, no answer survives in one chunk.
- Too large → embeddings get blurry, irrelevant text drowns the signal.
- Typical sweet spot for RAG: **500–1000 tokens with ~10–20% overlap.**

**Why metadata matters (cert + interview):**
- The canonical 7-step pipeline (`ingest → parse → chunk → embed → index → retrieve → generate`) hides metadata, but in any real system it's a **side-channel** that rides with each chunk through index → retrieve.
- In FSR: filtering by `ESN` / `serial` matters more than pure semantic similarity. Wrong-machine chunks look great but answer the wrong question.
- Databricks Vector Search calls these **"filterable columns"** — declare at index time, pass `filters=` at query time.
- For the POC we carry the *shape* (`doc_id`, `page`, `extra` dict for FSR-style fields). Real extraction (regex / LLM / hybrid) is a Phase 2 variant. FSR design ref: `implementation/design/fsr-pipeline-design.md`, `implementation/design/metadata-first-end-to-end-flow.drawio`.

**This notebook:**
1. Re-parse the PDF (silver runs standalone — no shared kernel state with bronze).
2. Build per-document metadata (`doc_id`, `extra` placeholder for FSR fields).
3. Baseline chunker: `RecursiveCharacterTextSplitter` with `tiktoken` for accurate token counts; metadata attached to each chunk.
4. Run 3 variants side by side; eyeball a sample chunk.
5. Persist for the next step (embed).

## Step 0 — Re-parse the source PDF

Same code as bronze; pasted here so silver can run on its own.
Output: `pages = [{"page": int, "text": str}, ...]`.

In [ ]:
from pathlib import Path
import fitz  # PyMuPDF

PDF_PATH = Path("/home/u560060992/dbx/ai-arch/sample-docs/fsr-sample-01.pdf")
assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"

doc = fitz.open(PDF_PATH)
pages = [{"page": i + 1, "text": doc.load_page(i).get_text()} for i in range(doc.page_count)]
doc.close()

print(f"pages: {len(pages)}, total chars: {sum(len(p['text']) for p in pages):,}")

## Step 1 — Token counter

We measure chunk size in **tokens**, not characters, because the embedding model bills/limits in tokens.

We use OpenAI's `cl100k_base` encoder via `tiktoken` as a stand-in. Different models tokenize differently, but `cl100k_base` is a reasonable proxy and what most LangChain examples use. For Databricks/Azure embedding models the actual tokenization may differ slightly — fine for sizing decisions.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    return len(enc.encode(text))

# sanity: tokens for the whole document
full_text = "\n\n".join(p["text"] for p in pages)
print(f"document tokens: {n_tokens(full_text):,}")

## Step 0.5 — Document metadata

Every chunk we emit will carry these fields. The shape matches what we'd want in production; the FSR-specific values (ESN, serial, doc date) are placeholders here — Phase 2 wires up real extraction.

**Cert language:**
- `doc_id` is the unit Vector Search uses for `delta_sync` indexing and for `filters` at query time.
- `extra` becomes additional **filterable columns** on the Delta source table when we go to Databricks.

**FSR design alignment:** see `metadata-first-end-to-end-flow.drawio` — metadata is a first-class column, not a string blob.

In [ ]:
import hashlib

def make_doc_metadata(pdf_path: Path) -> dict:
    """Build per-document metadata. POC: deterministic doc_id from filename;
    FSR-specific fields are placeholders to be filled in Phase 2 via regex/LLM extraction."""
    doc_id = hashlib.sha1(pdf_path.name.encode()).hexdigest()[:12]
    return {
        "doc_id": doc_id,
        "source_path": str(pdf_path),
        "source_name": pdf_path.name,
        # FSR-style filterable fields — placeholders for now, real extraction is Phase 2.
        "extra": {
            "esn": None,           # e.g. "810893"
            "serial": None,        # e.g. "SY0048230"
            "doc_date": None,      # e.g. "2024-08-15"
            "doc_type": "FSR",
        },
    }

doc_meta = make_doc_metadata(PDF_PATH)
print(doc_meta)

## Step 2 — Baseline chunker: `RecursiveCharacterTextSplitter`

**How it works:** tries to split on the highest-level separator that produces small-enough pieces — paragraph (`\n\n`) → line (`\n`) → sentence-ish (`. `) → word (`" "`) → character. Falls down the list only if pieces are still too big. This keeps semantically related text together when possible.

We tell it to *measure size in tokens* by passing `length_function=n_tokens`.

We chunk **per page** and tag each chunk with its page number, so downstream retrieval can cite a page.
(In Phase 2 we'll consider crossing page boundaries with a small overlap — but per-page is the cleanest baseline.)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pages(pages, doc_meta: dict, chunk_size: int, chunk_overlap: int) -> list[dict]:
    """Split pages and attach metadata (doc-level + page + chunk-position) to every chunk."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=n_tokens,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    out = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            out.append({
                # the text we'll embed
                "text": piece,
                # ── metadata (rides with chunk into the index) ──
                "doc_id":      doc_meta["doc_id"],
                "source_name": doc_meta["source_name"],
                "page":        p["page"],
                "chunk_idx":   i,                          # position within the page
                "chunk_id":    f"{doc_meta['doc_id']}_p{p['page']}_c{i}",
                "tokens":      n_tokens(piece),
                "extra":       doc_meta["extra"],          # FSR-specific filterable fields
            })
    return out

# baseline: 1000 tokens / 200 overlap (LangChain RAG default-ish)
chunks_baseline = chunk_pages(pages, doc_meta, chunk_size=1000, chunk_overlap=200)
print(f"baseline (1000/200): {len(chunks_baseline)} chunks")
print("sample chunk keys:", list(chunks_baseline[0].keys()))

## Step 3 — Compare 3 variants side by side

Just to build intuition for what knobs do what.

| variant       | chunk_size | overlap | when to prefer                                                |
| ------------- | ---------- | ------- | ------------------------------------------------------------- |
| small         | 300        | 50      | dense Q&A on facts; finer-grained retrieval                   |
| baseline      | 1000       | 200     | general RAG default                                           |
| large         | 1500       | 300     | long narrative content; fewer, richer chunks                  |

In [ ]:
import statistics

variants = {
    "small (300/50)":      chunk_pages(pages, doc_meta, 300, 50),
    "baseline (1000/200)": chunks_baseline,
    "large (1500/300)":    chunk_pages(pages, doc_meta, 1500, 300),
}

print(f"{'variant':<22} {'count':>6} {'min':>5} {'p50':>5} {'p95':>5} {'max':>5}")
print("-" * 52)
for name, ch in variants.items():
    sizes = [c["tokens"] for c in ch]
    p50 = int(statistics.median(sizes))
    p95 = int(sorted(sizes)[int(0.95 * (len(sizes) - 1))])
    print(f"{name:<22} {len(ch):>6} {min(sizes):>5} {p50:>5} {p95:>5} {max(sizes):>5}")

## Step 4 — Eyeball a chunk

Pick a baseline chunk roughly in the middle and read it. Looking for:
- Did the splitter break mid-sentence? (Bad — retrieval will lose context.)
- Are page headers/footers polluting every chunk? (Common; clean in Phase 2.)
- Is there enough self-contained meaning here that an embedding could represent it well?

In [ ]:
sample = chunks_baseline[len(chunks_baseline) // 2]
print(f"chunk_id: {sample['chunk_id']}  page: {sample['page']}  tokens: {sample['tokens']}")
print(f"doc_id:   {sample['doc_id']}  source: {sample['source_name']}")
print(f"extra:    {sample['extra']}")
print("-" * 60)
print(sample["text"])

## Step 5 — Persist baseline chunks for the next step

Save to JSON so the embed notebook (next) can load directly without re-parsing/re-chunking.

In [ ]:
import json

out_dir = Path("/home/u560060992/dbx/ai-arch/code/silver/_artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "chunks_baseline.json"
out_path.write_text(json.dumps(chunks_baseline, indent=2))
print(f"wrote {len(chunks_baseline)} chunks → {out_path}")

## Notes / parking lot for Phase 2

- **Real metadata extraction** — fill `extra.esn`, `extra.serial`, `extra.doc_date` for real. FSR pipeline does this with regex + LLM fallback; design ref: `implementation/design/fsr-pipeline-design.md`. Compare regex / LLM / hybrid as a Phase 2 variant (already on the tracker's Phase 2 grid).
- **Header/footer scrubbing** — strip repeated lines that appear on >X% of pages before chunking.
- **Char-soup pages** (the rotated/decorated mid-doc pages we saw in bronze) — current splitter happily produces garbage chunks from them. Phase 2: detect and route those pages to OCR or skip.
- **Cross-page chunks** — for narrative sections that flow across pages, per-page splitting cuts mid-thought. Try concatenating then splitting once, with page ranges tracked instead of single page numbers.
- **Structure-aware splitting** — markdown/heading-aware splitter for docs with clear section structure.
- **Semantic chunking** — embed-then-cluster approaches (later, only if baseline retrieval is poor).

## Cert tie-in (Week 1)

- Know the trade-off: **small vs large chunks** (recall/precision-style).
- Know `RecursiveCharacterTextSplitter` exists and what its separator hierarchy does.
- Know **overlap** is to preserve context across boundaries (typical 10–20%).
- Know chunk size is measured in **tokens** for embedding/serving cost reasons.
- Know **metadata = filterable columns** in Vector Search; declared at index creation, used as `filters` at query time.